# Training-init seed stability — cancer vs healthy (training + eval)

Retrains the two main healthy-vs-cancer leave-one-plate-out models — the cell-level
`Classifier` and the MIL `GatedAttention` — across 10 training-init seeds, and evaluates
each **identically** to the main notebooks (same `eval_model` fixed seeds and the same test
bagloaders). Only the training init seed varies: the global RNG is seeded *before* model
construction and passed to `train_model`, so both weight init and training order depend on it.

This is the heavy step (10 seeds × 16 plates × 2 models); run it once. It is **idempotent** —
models go to `SEED_OUTDIR` and per-plate predictions to `results/seed_stability/`, and anything
already present is skipped. The figure (S8d in `figure_S8.ipynb`) only reads the prediction CSVs
written here, so this notebook can be skipped when regenerating figures.

**Note on reproducibility:** even with every seed fixed, results are not bit-for-bit
reproducible — some PyTorch/cuDNN GPU kernels are non-deterministic and behaviour also depends
on the hardware and library versions. Re-running can therefore give slightly different numbers;
the point of this experiment is that the run-to-run variation is small, not that runs are identical.

In [1]:
import importlib
import util, models, training
importlib.reload(util); importlib.reload(models); importlib.reload(training)

from data import PlateDataset
from util import torch_random_choice
from models import Classifier, GatedAttention
from training import train_model

import os
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from tqdm import tqdm

device = 'cuda:0'

In [2]:
SEED_OUTDIR = '/ewsc/hschluet/models/pbmc5/seed_stability'
RESULT_DIR = 'results/seed_stability'
os.makedirs(SEED_OUTDIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

SEEDS = list(range(10))
PLATES = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])
CELL_ITERS, MIL_ITERS = 100_000, 50_000  # same as the main cell-level / MIL notebooks
AUG = T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), T.RandomHorizontalFlip(), T.RandomVerticalFlip()])


def seed_all(s):  # seed everything BEFORE model construction so weight init depends on the seed too
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [3]:
data = PlateDataset([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], load_masks=True)
cdf = data.info.groupby(['patient', 'time'])['cell'].count().reset_index()
cdf = cdf[cdf['cell'] > 100]
p01s = cdf[(cdf['time'] == 1) | (cdf['time'] == 0)]['patient'].unique()
pat_plates = data.info.groupby('plate')['patient'].unique()
pat_plate_map = data.info.groupby('patient')['plate'].unique()

  0%|          | 0/16 [00:00<?, ?it/s]

  6%|▋         | 1/16 [00:00<00:05,  2.63it/s]

 12%|█▎        | 2/16 [00:01<00:07,  1.76it/s]

 19%|█▉        | 3/16 [00:01<00:05,  2.47it/s]

 25%|██▌       | 4/16 [00:02<00:06,  1.76it/s]

...

 81%|████████▏ | 13/16 [00:07<00:01,  1.50it/s]

 88%|████████▊ | 14/16 [00:08<00:01,  1.50it/s]

 94%|█████████▍| 15/16 [00:08<00:00,  1.52it/s]

100%|██████████| 16/16 [00:09<00:00,  1.80it/s]

100%|██████████| 16/16 [00:09<00:00,  1.75it/s]

## Cell-level classifier — bagloaders, eval, train/eval routine

Same as `cell_level_models_same_architecture_as_mil.ipynb`, wrapped so the seed can vary.

In [4]:
def cell_train_bagloader(use_patients, use_plates, bag_size=50, use_times=(0, 1)):
    use_idx = torch.argwhere(torch.from_numpy(
        (data.info['plate'].isin(use_plates) & data.info['time'].isin(list(use_times))).values)).flatten()
    use_data = T.CenterCrop(28)(data.imgs[use_idx].to(device))
    plate_pats = np.concatenate(data.info.groupby(['plate'])['patient'].unique().loc[use_plates].values)
    up = use_patients[np.isin(use_patients, plate_pats)]
    pat_groups = data.info.groupby(['time'])['patient'].unique().map(lambda x: x[np.isin(x, up)])
    pat_healthy, pat_cancer = pat_groups[0], pat_groups[1]
    pat_lut = {p: torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == p).values)).flatten().to(device) for p in up}
    pat_min = min(len(pat_healthy), len(pat_cancer))
    while True:
        xs, labels = [], []
        for lab, pats in zip([0, 1], [pat_cancer, pat_healthy]):
            sub = np.random.choice(pats, size=pat_min, replace=False)
            for pat in sub:
                xs.append(use_data[torch_random_choice(pat_lut[pat], size=bag_size)])
                labels.append(lab * torch.ones(bag_size))
        labels = torch.cat(labels).float().to(device); xs = torch.cat(xs).float()
        rand_idx = torch.randperm(len(xs))
        for left in range(0, bag_size * 2 * pat_min, bag_size):
            yield xs[rand_idx[left:left + bag_size]], labels[rand_idx[left:left + bag_size]]


def cell_test_bagloader(use_plates, bag_size=100, use_times=(0, 1)):
    use_idx = torch.argwhere(torch.from_numpy(
        (data.info['plate'].isin(use_plates) & data.info['time'].isin(list(use_times))).values)).flatten()
    use_imgs = T.CenterCrop(28)(data.imgs[use_idx].to(device))
    for pat in p01s:
        if not np.isin(pat_plate_map[pat], use_plates).any():
            continue
        pat_idx = torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
        idx = torch_random_choice(pat_idx, size=bag_size)
        label = 1 if pat[0] == 'H' else 0
        yield use_imgs[idx].float(), torch.tensor(label).to(device), pat, 'healthy' if pat[0] == 'H' else 'cancer'


def cell_eval(model, loader):  # identical to eval_model in the cell-level notebook (incl. fixed eval seeds)
    model = model.to(device).eval()
    np.random.seed(1232412); torch.manual_seed(124514); torch.cuda.manual_seed_all(13513)
    L, P, I, PA, G = [], [], [], [], []
    for i, (bag, lab, pat, group) in enumerate(loader):
        with torch.no_grad():
            _, pred, z = model(bag.to(device))
        I.extend([i] * 100); PA.extend([pat] * 100); G.extend([group] * 100)
        L.extend([lab.item()] * 100); P.extend(pred.cpu().numpy())
    return pd.DataFrame({'lab': L, 'pred': P, 'i': I, 'pat': PA, 'group': G})


def run_cell(seed, plate):
    fname = f'seed{seed}_1_16_t01_healthy_cancer_without_plate_{plate}_by_cell_mil_architecture'
    seed_all(seed)
    model = Classifier()  # init depends on seed
    train_loader = cell_train_bagloader(p01s[~np.isin(p01s, pat_plates[plate])], PLATES[PLATES != plate])
    train_model(model, bag_loader=train_loader, num_iter=CELL_ITERS, lr=0.0001, device=device,
                fname=fname, plot=False, save_model=True, seed=seed, transform=AUG, bar=False, outdir=SEED_OUTDIR)
    model = Classifier().to(device)
    model.load_state_dict(torch.load(f'{SEED_OUTDIR}/{fname}_model.pt', weights_only=True, map_location=device))
    df = cell_eval(model, cell_test_bagloader([plate]))
    df['plate'] = plate; df['seed'] = seed
    return df

## Run cell-level across seeds

Heavy step — already-computed folds are skipped.

In [5]:
for seed in SEEDS:
    for plate in tqdm(PLATES, desc=f'cell seed {seed}'):
        out = f'{RESULT_DIR}/preds_cell_seed{seed}_plate{plate}.csv'
        if os.path.exists(out):
            continue
        run_cell(seed, plate).to_csv(out, index=False)
print('cell-level done')

cell seed 0:   0%|          | 0/16 [00:00<?, ?it/s]

cell seed 0: 100%|██████████| 16/16 [00:00<00:00, 3504.56it/s]

cell seed 1:   0%|          | 0/16 [00:00<?, ?it/s]

cell seed 1: 100%|██████████| 16/16 [00:00<00:00, 3947.58it/s]

cell seed 2:   0%|          | 0/16 [00:00<?, ?it/s]

...

cell seed 8:   0%|          | 0/16 [00:00<?, ?it/s]

cell seed 8: 100%|██████████| 16/16 [00:00<00:00, 4138.44it/s]

cell seed 9:   0%|          | 0/16 [00:00<?, ?it/s]

cell seed 9: 100%|██████████| 16/16 [00:00<00:00, 4553.46it/s]

cell-level done


## MIL model — bagloaders, eval, train/eval routine

Same as `healthy_vs_cancer_mil.ipynb`, wrapped so the seed can vary.

In [7]:
def mil_train_bagloader(use_patients, use_plates, bag_size=50, use_times=(0, 1)):
    use_idx = torch.argwhere(torch.from_numpy(
        (data.info['plate'].isin(use_plates) & data.info['time'].isin(list(use_times))).values)).flatten()
    use_data = T.CenterCrop(28)(data.imgs[use_idx].to(device))
    plate_pats = np.concatenate(data.info.groupby(['plate'])['patient'].unique().loc[use_plates].values)
    up = use_patients[np.isin(use_patients, plate_pats)]
    pat_groups = data.info.groupby(['time'])['patient'].unique().map(lambda x: x[np.isin(x, up)])
    pat_healthy, pat_cancer = pat_groups[0], pat_groups[1]
    pat_lut = {p: torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == p).values)).flatten().to(device) for p in up}
    while True:
        label = np.random.randint(2)
        pat = np.random.choice(pat_healthy if label == 1 else pat_cancer)
        xs = use_data[torch_random_choice(pat_lut[pat], size=bag_size)]
        yield xs.float(), torch.tensor(label).to(device)


def mil_test_bagloader(use_plates, bag_size=50, reps=5, use_times=(0, 1)):
    use_idx = torch.argwhere(torch.from_numpy(
        (data.info['plate'].isin(use_plates) & data.info['time'].isin(list(use_times))).values)).flatten()
    use_data = T.CenterCrop(28)(data.imgs[use_idx].to(device))
    use_masks = T.CenterCrop(28)(data.masks[use_idx].to(device))
    for pat in p01s:
        if not np.isin(pat_plate_map[pat], use_plates).any():
            continue
        pat_idx = torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
        for _ in range(reps):
            idx = torch_random_choice(pat_idx, size=bag_size)
            label = 1 if pat[0] == 'H' else 0
            yield use_data[idx].float(), use_masks[idx], torch.tensor(label).to(device), pat, 'healthy' if pat[0] == 'H' else 'cancer'


def mil_eval(model, loader, branches=5):  # identical to eval_model in the MIL notebook (incl. fixed eval seeds)
    model = model.to(device).eval()
    np.random.seed(1232412); torch.manual_seed(124514); torch.cuda.manual_seed_all(13513)
    L, P, I, PA, G = [], [], [], [], []
    for i, (bag, mask, lab, pat, group) in enumerate(loader):
        with torch.no_grad():
            _, pred, attn, rattn, z = model(bag.to(device))
        I.extend([i] * 50); PA.extend([pat] * 50); G.extend([group] * 50)
        L.extend([lab.item()] * 50); P.extend([pred.cpu().item()] * 50)
    return pd.DataFrame({'lab': L, 'pred': P, 'i': I, 'pat': PA, 'group': G})


def run_mil(seed, plate):
    fname = f'seed{seed}_1_16_healthy_without_plate_{plate}_mil'
    seed_all(seed)
    model = GatedAttention(branches=5)  # init depends on seed
    train_loader = mil_train_bagloader(p01s[~np.isin(p01s, pat_plates[plate])], PLATES[PLATES != plate])
    train_model(model, bag_loader=train_loader, num_iter=MIL_ITERS, lr=0.0001, device=device,
                fname=fname, plot=False, save_model=True, seed=seed, transform=AUG, bar=False, outdir=SEED_OUTDIR)
    model = GatedAttention(branches=5).to(device)
    model.load_state_dict(torch.load(f'{SEED_OUTDIR}/{fname}_model.pt', weights_only=True, map_location=device))
    df = mil_eval(model, mil_test_bagloader([plate]))
    df['plate'] = plate; df['seed'] = seed
    return df

## Run MIL across seeds

Heavy step — already-computed folds are skipped.

In [8]:
for seed in SEEDS:
    for plate in tqdm(PLATES, desc=f'mil seed {seed}'):
        out = f'{RESULT_DIR}/preds_mil_seed{seed}_plate{plate}.csv'
        if os.path.exists(out):
            continue
        run_mil(seed, plate).to_csv(out, index=False)
print('MIL done')

mil seed 0:   0%|          | 0/16 [00:00<?, ?it/s]

mil seed 0: 100%|██████████| 16/16 [00:00<00:00, 3424.80it/s]

mil seed 1:   0%|          | 0/16 [00:00<?, ?it/s]

mil seed 1: 100%|██████████| 16/16 [00:00<00:00, 3924.72it/s]

mil seed 2:   0%|          | 0/16 [00:00<?, ?it/s]

...

mil seed 8:   0%|          | 0/16 [00:00<?, ?it/s]

mil seed 8: 100%|██████████| 16/16 [00:00<00:00, 3913.51it/s]

mil seed 9:   0%|          | 0/16 [00:00<?, ?it/s]

mil seed 9: 100%|██████████| 16/16 [00:00<00:00, 4075.60it/s]

MIL done


In [9]:
n = len([f for f in os.listdir(RESULT_DIR) if f.startswith('preds_')])
print(f'{n} per-(model, seed, plate) prediction files in {RESULT_DIR}  (expect 10 seeds x 16 plates x 2 = 320)')

320 per-(model, seed, plate) prediction files in results/seed_stability  (expect 10 seeds x 16 plates x 2 = 320)
